# Figure 4 · Workload runtime distribution

Kernel density of log-scaled per-query runtimes from each workload's execution run, saved to
`FIGURES_DIR/runtimes.pdf`. DiverSQL is shown with feedback only (W/ FB), as in the paper.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator
from scipy.stats import gaussian_kde

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
for _p in (str(ROOT), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from config import FIGURES_DIR
from workloads import PRETTY, load_runtimes

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Legend order and colours as in the paper.
LEGS = {
    "tpcds_ref": "#9ca3af",
    "tpcds_sqlbarber": "#a855f7",
    "tpcds_bootstrapping_lcm": "#6366f1",
    "tpcds_sql_factory": "#0891b2",
    "tpcds_e2etune": "#059669",
    "tpcds_sqlstorm": "#db2777",
    "tpcds_diversql": "#ea580c",
}
REF, OURS = "tpcds_ref", "tpcds_diversql"
LABELS = {**PRETTY, REF: "TPC-DS"}

# The TPC-DS curve uses every query of its 1,208-query run (Table 2 clips it to q1-q1000).
RUNTIMES = {w: load_runtimes(w) for w in LEGS}
for w, v in RUNTIMES.items():
    print(f"{LABELS[w]:20s} {len(v):5d} runtimes")

In [ ]:
NICE_TIME_TICKS = [(1e-3, "1 ms"), (1e-2, "10 ms"), (1e-1, "100 ms"), (1, "1 s"), (10, "10 s"),
                   (60, "1 min"), (600, "10 min"), (3600, "1 hr")]
font_size = 15.5
font_size_leg = 11.85

fig, ax = plt.subplots(figsize=(8.2, 3.5))

all_vals = np.concatenate(list(RUNTIMES.values()))
log_lo = min(np.log10(all_vals.min()), np.log10(1e-3))  # always start the axis at 1 ms
log_hi = np.log10(all_vals.max())
log_grid = np.linspace(log_lo, log_hi, 400)
grid = 10 ** log_grid

for w, color in LEGS.items():
    density = gaussian_kde(np.log10(RUNTIMES[w]))(log_grid)
    ax.plot(
        grid, density,
        linewidth=2.6 if w == OURS else 1.3,
        linestyle="--" if w == REF else "-",
        color=color,
        label=LABELS[w],
        zorder=5 if w == OURS else (4 if w == REF else 3),
    )

ax.set_xscale("log")
ax.xaxis.set_minor_locator(NullLocator())
ticks = [t for t, _ in NICE_TIME_TICKS if log_lo <= np.log10(t) <= log_hi]
ax.set_xticks(ticks)
ax.set_xticklabels([label for t, label in NICE_TIME_TICKS if t in ticks])
ax.set_xlim(10 ** log_lo, 10 ** log_hi)
ax.set_xlabel("Runtime (Log-Scale)", fontsize=font_size)
ax.set_ylabel("Density", fontsize=font_size)
ax.grid(linestyle="-", alpha=0.5)
ax.legend(fontsize=font_size_leg, labelspacing=0.125, handlelength=1.5, handletextpad=0.5,
          borderpad=0.3, borderaxespad=0.3)
ax.tick_params(axis="x", labelsize=font_size, pad=2)
ax.tick_params(axis="y", labelsize=font_size, pad=2)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "runtimes.pdf")
fig.savefig(FIGURES_DIR / "runtimes.png")
plt.show()